In [1]:
import os
import boto3
from sagemaker import get_execution_role
import time
from pprint import pprint
import shutil

sagemaker.config INFO - Not applying SDK defaults from location: /etc/xdg/sagemaker/config.yaml
sagemaker.config INFO - Not applying SDK defaults from location: /home/ec2-user/.config/sagemaker/config.yaml


### Constants

In [2]:
# name
str_function_name = 'genxii-ad-update-feats'

### 1. Create container

### Create ```Dockerfile```

In [3]:
%%writefile Dockerfile

FROM public.ecr.aws/lambda/python:3.8

# update pip
RUN pip install --upgrade pip

# install dependencies from project folder
COPY requirements.txt  .
RUN  pip3 install -r requirements.txt --target "${LAMBDA_TASK_ROOT}"

# copy function code
COPY lambda_function.py ${LAMBDA_TASK_ROOT}

# Set the CMD to your handler (could also be done as a parameter override outside of the Dockerfile)
CMD ["lambda_function.lambda_handler"] 

Writing Dockerfile


### Write ```requirements.txt```

In [4]:
%%writefile requirements.txt

pyarrow==9.0.0
fsspec==2022.10.0
s3fs==2022.10.0

pandas==1.2.4

Writing requirements.txt


### Write ```lambda_function.py```

In [5]:
%%writefile lambda_function.py

import pandas as pd

# lambda handler
def lambda_handler(event, context):
    # constants
    str_project = '20231010-gen-xii'
    str_model = '01_ad'
    
    # load output from concat sensitivity
    print('Loading output from sensitivity analysis concatenation...')
    str_filename = 'df_sensitivity.csv'
    str_uri = f's3://{str_project}/{str_model}/02_model/02_model/05_lambda_concat_sensitivity/{str_filename}'
    df = pd.read_csv(str_uri)
    
    # get the top feature
    print('Getting feature that helps the model most once removed...')
    str_col = df['feature'].iloc[0]
    
    # import the features to drop
    print('Importing features to drop...')
    str_filename = 'df_feats_to_drop.csv'
    str_uri = f's3://{str_project}/{str_model}/02_model/02_model/01_lambda_get_starting_feats/{str_filename}'
    list_cols_drop = list(pd.read_csv(str_uri)['feature'])
    
    # append
    print(f'Appending {str_col} to {str_uri}...')
    list_cols_drop.append(str_col)
    
    # create df and upload to s3
    print(f'Creating data frame and uploading to {str_uri}...')
    df = pd.DataFrame({'feature': list_cols_drop})
    df.to_csv(str_uri, index=False)

Writing lambda_function.py


### Build image and push to ECR

In [6]:
%%sh

# name the image
image=genxii-ad-update-feats

# build image
docker build -t ${image} .

# get region
region=$(aws configure get region)
region=${region:-us-west-2}

# get account
account=$(aws sts get-caller-identity --query Account --output text)

# get full name
fullname="${account}.dkr.ecr.${region}.amazonaws.com/${image}:latest"

# get login command and execute it
aws ecr get-login-password --region "${region}" | docker login --username AWS --password-stdin "${account}".dkr.ecr."${region}".amazonaws.com

# create repository in ECR
aws ecr create-repository --repository-name "${image}" --image-scanning-configuration scanOnPush=true --image-tag-mutability MUTABLE

# tag image
docker tag  ${image} ${fullname}

# push image to ECR   
docker push ${fullname}

Sending build context to Docker daemon  29.18kB
Step 1/6 : FROM public.ecr.aws/lambda/python:3.8
 ---> 3cd81ffec4d9
Step 2/6 : RUN pip install --upgrade pip
 ---> Using cache
 ---> 1fa305cd0a48
Step 3/6 : COPY requirements.txt  .
 ---> Using cache
 ---> b5f93c0f273f
Step 4/6 : RUN  pip3 install -r requirements.txt --target "${LAMBDA_TASK_ROOT}"
 ---> Using cache
 ---> 45f967c05773
Step 5/6 : COPY lambda_function.py ${LAMBDA_TASK_ROOT}
 ---> Using cache
 ---> 47391984f9eb
Step 6/6 : CMD ["lambda_function.lambda_handler"]
 ---> Using cache
 ---> f8306a28ab8c
Successfully built f8306a28ab8c
Successfully tagged genxii-ad-update-feats:latest


WARNING! Your password will be stored unencrypted in /home/ec2-user/.docker/config.json.
Configure a credential helper to remove this warning. See
https://docs.docker.com/engine/reference/commandline/login/#credentials-store



Login Succeeded



An error occurred (RepositoryAlreadyExistsException) when calling the CreateRepository operation: The repository with name 'genxii-ad-update-feats' already exists in the registry with id '836690756591'


The push refers to repository [836690756591.dkr.ecr.us-west-2.amazonaws.com/genxii-ad-update-feats]
0aff4545474f: Preparing
29a64d0015ae: Preparing
e9fb2647502c: Preparing
3f97a2d36016: Preparing
e92756f7b561: Preparing
4fe51bf0bf5c: Preparing
fbbd8c1e2ec1: Preparing
fe2359fe88f2: Preparing
e703f2e518cc: Preparing
97a787951169: Preparing
fbbd8c1e2ec1: Waiting
fe2359fe88f2: Waiting
e703f2e518cc: Waiting
4fe51bf0bf5c: Waiting
97a787951169: Waiting
e92756f7b561: Layer already exists
0aff4545474f: Layer already exists
3f97a2d36016: Layer already exists
29a64d0015ae: Layer already exists
e9fb2647502c: Layer already exists
4fe51bf0bf5c: Layer already exists
fbbd8c1e2ec1: Layer already exists
fe2359fe88f2: Layer already exists
97a787951169: Layer already exists
e703f2e518cc: Layer already exists
latest: digest: sha256:fc9b896d10ea7a5af92183ddc5f0577263ecc68464d536717de8b976d3a12d1c size: 2419


### 2. Create lambda function from image

In [7]:
# initialize class
cls_client_lambda = boto3.client('lambda')

In [8]:
# get role
str_role = get_execution_role()
print(f'Role: {str_role}')

Role: arn:aws:iam::836690756591:role/risk-ops-role


In [9]:
# delete it if it exists
try:
    dict_response = cls_client_lambda.delete_function(
        FunctionName=str_function_name,
    )
    pprint(dict_response)
except:
    pass

{'ResponseMetadata': {'HTTPHeaders': {'connection': 'keep-alive',
                                      'content-type': 'application/json',
                                      'date': 'Mon, 22 Apr 2024 19:46:06 GMT',
                                      'x-amzn-requestid': '9b94e0e1-d8d0-4303-b198-cc542b9be25d'},
                      'HTTPStatusCode': 204,
                      'RequestId': '9b94e0e1-d8d0-4303-b198-cc542b9be25d',
                      'RetryAttempts': 0}}


In [10]:
# create function
str_image_uri = '836690756591.dkr.ecr.us-west-2.amazonaws.com/genxii-ad-update-feats:latest' # this must match what we name the image above
dict_response = cls_client_lambda.create_function(
    FunctionName=str_function_name,
    Role=str_role,
    Code={
        'ImageUri': str_image_uri,
    },
    Timeout=60,
    MemorySize=512,
    Publish=True,
    PackageType='Image',
    Architectures=[
        'x86_64',
    ],
    EphemeralStorage={
        'Size': 512,
    },
)
pprint(dict_response)
time.sleep(40)

{'Architectures': ['x86_64'],
 'CodeSha256': 'fc9b896d10ea7a5af92183ddc5f0577263ecc68464d536717de8b976d3a12d1c',
 'CodeSize': 0,
 'Description': '',
 'EphemeralStorage': {'Size': 512},
 'FunctionArn': 'arn:aws:lambda:us-west-2:836690756591:function:genxii-ad-update-feats',
 'FunctionName': 'genxii-ad-update-feats',
 'LastModified': '2024-04-22T19:46:06.757+0000',
 'LoggingConfig': {'LogFormat': 'Text',
                   'LogGroup': '/aws/lambda/genxii-ad-update-feats'},
 'MemorySize': 512,
 'PackageType': 'Image',
 'ResponseMetadata': {'HTTPHeaders': {'connection': 'keep-alive',
                                      'content-length': '1198',
                                      'content-type': 'application/json',
                                      'date': 'Mon, 22 Apr 2024 19:46:07 GMT',
                                      'x-amzn-requestid': '1dff0101-870e-4268-a8c5-097c1021afdc'},
                      'HTTPStatusCode': 201,
                      'RequestId': '1dff0101-870e-42

### Clean-up

In [11]:
for str_file in ['Dockerfile', 'lambda_function.py', 'requirements.txt']:
    os.remove(str_file)